In [ ]:
# ..... paralog list across species ..... #
# cross-primate comparisons

In [2]:
library(data.table)
library(dplyr)

In [3]:
# get list of paralogs 
tab1 = read.delim('/vault/suresh/consensus_interneuron_taxonomy/homologs/human_paralogs.txt', sep = '\t')
tab1 <- tab1[(tab1[,1]!='' & tab1[,2]!='' & tab1[,4]!='' & tab1[,4]!='gene_split'),]
tab1 <- tab1[!duplicated(tab1),]
colnames(tab1)[1:2] = c('sp1_gene', 'sp2_gene')

# human - NHP
paralog_df1 = paste0(unlist(tab1$sp1_gene), '_', unlist(tab1$sp2_gene))

# NHP - human
paralog_df2 = paralog_df1

head(paralog_df1)

[1] "MT-ND2_MT-ND4" "MT-ND2_MT-ND5" "MT-ND4_MT-ND5" "MT-ND4_MT-ND2"
[5] "MT-ND5_MT-ND4" "MT-ND5_MT-ND2"

In [4]:
# species in Ma et al
spelist1 = c('human', 'chimp', 'macaque', 'marmoset')

# species in Caglayan et al
spelist2 = c('human', 'chimp', 'macaque')

In [69]:
currstudy = 'caglayan'

if(currstudy=='sestan'){
    spe_combos = combn(spelist1, 2)
}else{
    spe_combos = combn(spelist2, 2)
}
spe_combos

human,human,chimp
chimp,macaque,macaque


In [82]:
# get marker lists
currid = 3
sp1 = spe_combos[1,currid]
sp2 = spe_combos[2,currid]

tab2 = fread(paste0(currstudy, '_', sp1, '_de_novo_cluster_markers.csv.gz'))
ctypes2 = unique(tab2$cell_type)
tab2[1:2,]

tab3 = fread(paste0(currstudy, '_', sp2, '_de_novo_cluster_markers.csv.gz'))
ctypes3 = unique(tab3$cell_type)
tab3[1:2,]

group,cell_type,gene,fold_change,auroc,log_fdr,population_size,population_fraction,average_expression,se_expression,detection_rate,fold_change_detection,precision,recall
<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
all,0,SLC9A9,5.939651,0.9018604,-523.8011,593,0.07579243,1000.911,27.54975,0.9443508,2.633997,0.17789072,0.9443508
all,0,DPP10,3.631426,0.8889016,-490.4817,593,0.07579243,9518.901,180.87971,0.9966273,1.101143,0.08283111,0.9966273


group,cell_type,gene,fold_change,auroc,log_fdr,population_size,population_fraction,average_expression,se_expression,detection_rate,fold_change_detection,precision,recall
<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
all,0,SLC9A9,4.194367,0.8907471,-488.5573,588,0.07824351,1340.0891,31.55378,0.9659864,1.873169,0.1372976,0.9659864
all,0,RGS5,8.538241,0.8621041,-420.0706,588,0.07824351,448.0416,17.49235,0.8231293,6.780568,0.3669447,0.8231293


In [83]:
# make 2 sets, one for each direction
# sp1 - sp2
paralog_df1 = data.frame(genes = paste0(unlist(tab1[,1]), '_', unlist(tab1[,2])),
                        LCA = tab1[,3], homology_type = tab1[,4])
head(paralog_df1, n = 3)

# sp2 - sp1
paralog_df2 = paralog_df1

,genes,LCA,homology_type
,<chr>,<chr>,<chr>
1,MT-ND2_MT-ND4,Bilateria,other_paralog
2,MT-ND2_MT-ND5,Bilateria,other_paralog
3,MT-ND4_MT-ND5,Bilateria,other_paralog


In [84]:
# get top 100 markers
tab2$rank = rep(1:(dim(tab2)[1]/length(ctypes2)), length(ctypes2))
top_markers1 = tab2[(tab2$rank<=100),]
tab3$rank = rep(1:(dim(tab3)[1]/length(ctypes3)), length(ctypes3))
top_markers2 = tab3[(tab3$rank<=100),]

In [85]:
combos = rbind(rep(ctypes2, each = length(ctypes3)), rep(ctypes3, length(ctypes2)))
dim(combos)
combos[1:2,1:4]

[1]   2 841

0,0,0,0
0,1,10,11


In [86]:
# get details of lca, spec scores
get_paralog_details <- function(vec1, vec2, pmat){
    mat1 = data.frame(g1 = rep(vec1, each = length(vec2)), 
                      g2 = rep(vec2, length(vec1)), homolog = NA, LCA = NA, homology_type = NA)
    glist = paste0(mat1[,1], '_', mat1[,2])
    temp2 = match(glist, pmat[,1])
    
    mat1$homolog = !is.na(temp2)
    mat1$LCA = pmat[temp2,2]
    mat1$homology_type = pmat[temp2,3]
    return(mat1)
}

In [87]:
newdf2 = c()
pb = txtProgressBar(min = 0, max = dim(combos)[2], initial = 0)

for(ii in 1:dim(combos)[2]){
    ctype1 = combos[1,ii]
    ctype2 = combos[2,ii]
    m1 = top_markers1$gene[top_markers1$cell_type==ctype1]
    m2 = top_markers2$gene[top_markers2$cell_type==ctype2]
    
    temp = intersect(m1, m2)
    
    if(length(temp)){
        rem_genes1 = setdiff(m1, temp)
        rem_genes2 = setdiff(m2, temp)

        # get orthologs
        list_ortholog = data.frame(g1 = temp, g2 = temp,
                                   homolog = TRUE, LCA = NA,
                                   homology_type = 'ortholog_one2one')   
        list_ortholog$cluster1 = ctype1
        list_ortholog$cluster2 = ctype2
        newdf2 = rbind(newdf2, list_ortholog)
    }else{
        rem_genes1 = m1
        rem_genes2 = m2
    }
    
    if(length(rem_genes1)){
        list1 = get_paralog_details(rem_genes1, m2, paralog_df1)
        list1$cluster1 = ctype1
        list1$cluster2 = ctype2
        newdf2 = rbind(newdf2, list1)
    }
    if(length(rem_genes2)){
        list2 = get_paralog_details(rem_genes2, m1, paralog_df2) 
        temp_list2 <- list2[,c(2,1,3:5)]
        colnames(temp_list2) = colnames(list2)
        list2 <- temp_list2
        
        list2$cluster1 = ctype1
        list2$cluster2 = ctype2        
        newdf2 = rbind(newdf2, list2)
    }
             
    setTxtProgressBar(pb, ii)
}

newdf2 <- newdf2[newdf2$homolog==TRUE,]
newdf2$homology[newdf2$homology_type!='ortholog_one2one'] = 'ortholog_many2many'

# removing LCA column since comparing among human paralogs
newdf2 <- newdf2[,-match(c('LCA', 'homology_type'), colnames(newdf2))]

newdf2 <- newdf2[!(duplicated(newdf2)),]
newdf2$species1 = sp1
newdf2$species2 = sp2
newdf2$primate_study = currstudy

dim(newdf2)
newdf2[1:5,]

path_to_save = paste0('/vault/suresh/consensus_interneuron_taxonomy/', currstudy, '/')
write.table(newdf2, file = paste0(path_to_save, sp1, '_', sp2, '_', currstudy, '_marker_paralogs_list.csv'), 
            sep = ',', row.names = F, col.names = T, quote = F)

[1] 47670     9

,g1,g2,homolog,cluster1,cluster2,homology,species1,species2,primate_study
,<chr>,<chr>,<lgl>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
1,SLC9A9,SLC9A9,TRUE,0,0,ortholog_one2one,chimp,macaque,caglayan
2,DPP10,DPP10,TRUE,0,0,ortholog_one2one,chimp,macaque,caglayan
3,ADAMTS17,ADAMTS17,TRUE,0,0,ortholog_one2one,chimp,macaque,caglayan
4,TMEM132C,TMEM132C,TRUE,0,0,ortholog_one2one,chimp,macaque,caglayan
5,ERBB4,ERBB4,TRUE,0,0,ortholog_one2one,chimp,macaque,caglayan


In [50]:
# newdf2 = read.delim('mouse_human_marker_paralogs_list.csv', sep = ',')
newdf2[newdf2$cluster1=='17' & newdf2$cluster2=='17',]

,g1,g2,homolog,cluster1,cluster2,homology,species1,species2,primate_study
,<chr>,<chr>,<lgl>,<int>,<int>,<chr>,<chr>,<chr>,<chr>
4233748,THSD7B,THSD7B,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233749,MIR99AHG,MIR99AHG,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233750,ZBTB20,ZBTB20,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233751,DLX6-AS1,DLX6-AS1,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233752,GRM7,GRM7,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233753,GALNT13,GALNT13,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233754,DACH2,DACH2,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233755,PLD5,PLD5,TRUE,17,17,ortholog_one2one,human,marmoset,sestan
4233756,ADARB2,ADARB2,TRUE,17,17,ortholog_one2one,human,marmoset,sestan


In [23]:
length(intersect(top_markers1$gene[top_markers1$cell_type==17],
top_markers2$gene[top_markers2$cell_type==17]))

[1] 78

In [10]:
# mapping mouse subclass vs sestan cluster
mapdf = read.delim('mouse_primate_Caglayan_mapping.csv', sep = ',')
tail(mapdf)

,cluster,subclass,class
,<int>,<chr>,<chr>
21,20,Sncg,CGE
22,8,Chandelier,MGE
23,11,Lamp5 Lhx6,CGE
24,25,Sst Chodl,MGE
25,1,Lamp5,CGE
26,15,Lamp5,CGE


In [11]:
# distance betn clusters by nbd
newdf2$nbd1 = mapdf$class[match(newdf2$cluster1, mapdf$subclass)]
newdf2$nbd2 = mapdf$class[match(newdf2$cluster2, mapdf$cluster)]

newdf2$sub1 = newdf2$cluster1
newdf2$sub2 = mapdf$subclass[match(newdf2$cluster2, mapdf$cluster)]

newdf2$dist = NA
newdf2$dist[newdf2$nbd1 != newdf2$nbd2] = 2
newdf2$dist[newdf2$nbd1 == newdf2$nbd2] = 1
newdf2$dist[newdf2$sub1 == newdf2$sub2] = 0

newdf2[1:2,]

,g1,g2,paralog,LCA,homology_type,cluster1,cluster2,nbd1,nbd2,sub1,sub2,dist
,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,PRKG1,PRKG1,TRUE,Eutheria,ortholog_one2one,Chandelier,0,MGE,MGE,Chandelier,Pvalb,1
2,TMEM108,TMEM108,TRUE,Eutheria,ortholog_one2one,Chandelier,0,MGE,MGE,Chandelier,Pvalb,1


In [12]:
table(tab1[,3])
table(newdf2$LCA)


                                Amniota                               Bilateria 
                                  19283                                  113006 
                          Boreoeutheria                              Catarrhini 
                                   2101                                    1622 
                               Chordata                        Euarchontoglires 
                                 275423                                   16436 
                           Euteleostomi                                Eutheria 
                                  88784                                   31641 
                                 Glires                           Gnathostomata 
                                     98                                   83249 
                            Haplorrhini                               Hominidae 
                                     48                                     281 
                           


         Amniota        Bilateria    Boreoeutheria         Chordata 
               8             3843               98              182 
Euarchontoglires     Euteleostomi         Eutheria    Gnathostomata 
             680               11              147               43 
    Opisthokonta       Vertebrata 
             511              219 

In [17]:
agegrp = 'Bilateria'
phyper(sum(newdf2$LCA==agegrp), sum(tab1[,3]==agegrp)/2, 
       (dim(tab1)[1] - sum(tab1[,3]==agegrp))/2, dim(newdf2)[1], lower.tail = F)

[1] 0